<a href="https://colab.research.google.com/github/sahitani/Gen-AI-Project/blob/main/titanic_competition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1 — Setup and load data

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

# Load the uploaded datasets
train_df = pd.read_csv('/content/sample_data/Titanic_train.csv')
test_df = pd.read_csv('/content/sample_data/Titanic_test.csv')

print("Train data shape:", train_df.shape)
print("Test data shape: ", test_df.shape)

print("\nFirst 3 rows of training data:")
display(train_df.head(3))

print("\nColumns in training data:")
print(train_df.columns.tolist())

Train data shape: (891, 12)
Test data shape:  (418, 11)

First 3 rows of training data:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S



Columns in training data:
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


In [2]:
# Cell 2 — Basic inspection of training data

print("=" * 60)
print("MISSING VALUES IN TRAIN DATA")
print("=" * 60)
missing_train = train_df.isnull().sum()
missing_pct_train = (missing_train / len(train_df) * 100).round(2)
missing_summary_train = pd.DataFrame({
    'Missing Count': missing_train,
    'Missing %': missing_pct_train
})
print(missing_summary_train[missing_summary_train['Missing Count'] > 0])

print("\n" + "=" * 60)
print("MISSING VALUES IN TEST DATA")
print("=" * 60)
missing_test = test_df.isnull().sum()
missing_pct_test = (missing_test / len(test_df) * 100).round(2)
missing_summary_test = pd.DataFrame({
    'Missing Count': missing_test,
    'Missing %': missing_pct_test
})
print(missing_summary_test[missing_summary_test['Missing Count'] > 0])

print("\n" + "=" * 60)
print("TARGET DISTRIBUTION (Survived)")
print("=" * 60)
print(train_df['Survived'].value_counts())
print(f"\nSurvival rate: {train_df['Survived'].mean()*100:.1f}%")

print("\n" + "=" * 60)
print("NUMERIC FEATURES SUMMARY")
print("=" * 60)
print(train_df.describe().round(2))

MISSING VALUES IN TRAIN DATA
          Missing Count  Missing %
Age                 177      19.87
Cabin               687      77.10
Embarked              2       0.22

MISSING VALUES IN TEST DATA
       Missing Count  Missing %
Age               86      20.57
Fare               1       0.24
Cabin            327      78.23

TARGET DISTRIBUTION (Survived)
Survived
0    549
1    342
Name: count, dtype: int64

Survival rate: 38.4%

NUMERIC FEATURES SUMMARY
       PassengerId  Survived  Pclass     Age   SibSp   Parch    Fare
count       891.00    891.00  891.00  714.00  891.00  891.00  891.00
mean        446.00      0.38    2.31   29.70    0.52    0.38   32.20
std         257.35      0.49    0.84   14.53    1.10    0.81   49.69
min           1.00      0.00    1.00    0.42    0.00    0.00    0.00
25%         223.50      0.00    2.00   20.12    0.00    0.00    7.91
50%         446.00      0.00    3.00   28.00    0.00    0.00   14.45
75%         668.50      1.00    3.00   38.00    1.00    0.

In [3]:
# Cell 3 — Survival rates broken down by categorical features

print("=" * 60)
print("SURVIVAL RATE BY SEX")
print("=" * 60)
sex_survival = train_df.groupby('Sex')['Survived'].agg(['mean', 'count']).round(3)
sex_survival.columns = ['Survival Rate', 'Total Passengers']
print(sex_survival)

print("\n" + "=" * 60)
print("SURVIVAL RATE BY PASSENGER CLASS")
print("=" * 60)
class_survival = train_df.groupby('Pclass')['Survived'].agg(['mean', 'count']).round(3)
class_survival.columns = ['Survival Rate', 'Total Passengers']
print(class_survival)

print("\n" + "=" * 60)
print("SURVIVAL RATE BY EMBARKED")
print("=" * 60)
embarked_survival = train_df.groupby('Embarked')['Survived'].agg(['mean', 'count']).round(3)
embarked_survival.columns = ['Survival Rate', 'Total Passengers']
print(embarked_survival)

print("\n" + "=" * 60)
print("SURVIVAL RATE BY SEX AND CLASS COMBINED")
print("=" * 60)
combined_survival = train_df.groupby(['Sex', 'Pclass'])['Survived'].agg(['mean', 'count']).round(3)
combined_survival.columns = ['Survival Rate', 'Total Passengers']
print(combined_survival)

print("\n" + "=" * 60)
print("SURVIVAL RATE BY FAMILY SIZE")
print("=" * 60)
train_df['family_size_temp'] = train_df['SibSp'] + train_df['Parch'] + 1
family_survival = train_df.groupby('family_size_temp')['Survived'].agg(['mean', 'count']).round(3)
family_survival.columns = ['Survival Rate', 'Total Passengers']
print(family_survival)
train_df = train_df.drop('family_size_temp', axis=1)  # cleanup

SURVIVAL RATE BY SEX
        Survival Rate  Total Passengers
Sex                                    
female          0.742               314
male            0.189               577

SURVIVAL RATE BY PASSENGER CLASS
        Survival Rate  Total Passengers
Pclass                                 
1               0.630               216
2               0.473               184
3               0.242               491

SURVIVAL RATE BY EMBARKED
          Survival Rate  Total Passengers
Embarked                                 
C                 0.554               168
Q                 0.390                77
S                 0.337               644

SURVIVAL RATE BY SEX AND CLASS COMBINED
               Survival Rate  Total Passengers
Sex    Pclass                                 
female 1               0.968                94
       2               0.921                76
       3               0.500               144
male   1               0.369               122
       2               0.

In [4]:
# Cell 4 — Combine datasets and engineer features

# Save target variable separately
y_train = train_df['Survived']

# Drop target from train so train and test have same columns
train_features = train_df.drop('Survived', axis=1)

# Add a marker column to track which is train vs test
train_features['source'] = 'train'
test_df['source'] = 'test'

# Combine into one dataframe
combined = pd.concat([train_features, test_df], axis=0, ignore_index=True)
print(f"Combined dataset shape: {combined.shape}")

# ============================================
# FEATURE ENGINEERING
# ============================================

# 1. Family size and is_alone
combined['family_size'] = combined['SibSp'] + combined['Parch'] + 1
combined['is_alone'] = (combined['family_size'] == 1).astype(int)

print(f"\nfamily_size distribution:")
print(combined['family_size'].value_counts().sort_index())

# 2. Extract Title from Name
# Names look like "Braund, Mr. Owen Harris"
# We extract whatever comes between ", " and ". "
combined['title'] = combined['Name'].str.extract(r' ([A-Za-z]+)\.')

print(f"\nRaw titles found:")
print(combined['title'].value_counts())

# Group rare titles into "Rare" category
title_mapping = {
    'Mr': 'Mr',
    'Miss': 'Miss',
    'Mrs': 'Mrs',
    'Master': 'Master',
    'Mlle': 'Miss',     # French Miss
    'Ms': 'Miss',
    'Mme': 'Mrs',       # French Mrs
}

# Apply mapping; everything not in mapping becomes "Rare"
combined['title'] = combined['title'].map(title_mapping).fillna('Rare')

print(f"\nFinal title categories:")
print(combined['title'].value_counts())

# 3. has_cabin binary flag
combined['has_cabin'] = combined['Cabin'].notna().astype(int)
print(f"\nhas_cabin distribution:")
print(combined['has_cabin'].value_counts())

# 4. Drop columns we will not use
columns_to_drop = ['Name', 'Ticket', 'Cabin', 'PassengerId']
# Note: keeping PassengerId for now to remerge for submission later
combined_passenger_ids = combined['PassengerId'].copy()  # save for later
combined = combined.drop(columns=['Name', 'Ticket', 'Cabin'])

print(f"\nCombined shape after engineering: {combined.shape}")
print(f"\nColumns now: {combined.columns.tolist()}")

print("\nFirst 3 rows of  data:")
display(combined.head(3))

Combined dataset shape: (1309, 12)

family_size distribution:
family_size
1     790
2     235
3     159
4      43
5      22
6      25
7      16
8       8
11     11
Name: count, dtype: int64

Raw titles found:
title
Mr          757
Miss        260
Mrs         197
Master       61
Rev           8
Dr            8
Col           4
Major         2
Mlle          2
Ms            2
Mme           1
Don           1
Sir           1
Lady          1
Capt          1
Countess      1
Jonkheer      1
Dona          1
Name: count, dtype: int64

Final title categories:
title
Mr        757
Miss      264
Mrs       198
Master     61
Rare       29
Name: count, dtype: int64

has_cabin distribution:
has_cabin
0    1014
1     295
Name: count, dtype: int64

Combined shape after engineering: (1309, 13)

Columns now: ['PassengerId', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'source', 'family_size', 'is_alone', 'title', 'has_cabin']

First 3 rows of  data:


,PassengerId,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,source,family_size,is_alone,title,has_cabin
0,1,3,male,22.0,1,0,7.2500,S,train,2,0,Mr,0
1,2,1,female,38.0,1,0,71.2833,C,train,2,0,Mrs,1
2,3,3,female,26.0,0,0,7.9250,S,train,1,1,Miss,0


In [5]:
# Cell 5 — Handle missing values and encode categoricals

# ============================================
# HANDLE MISSING VALUES
# ============================================

# 1. Age — fill with median
age_median = combined['Age'].median()
combined['Age'] = combined['Age'].fillna(age_median)
print(f"Age missing values filled with median: {age_median}")

# 2. Embarked — fill with mode (most common value)
embarked_mode = combined['Embarked'].mode()[0]
combined['Embarked'] = combined['Embarked'].fillna(embarked_mode)
print(f"Embarked missing values filled with mode: {embarked_mode}")

# 3. Fare — fill with median
fare_median = combined['Fare'].median()
combined['Fare'] = combined['Fare'].fillna(fare_median)
print(f"Fare missing values filled with median: {fare_median}")

# Verify no missing values remain
print("\nMissing values remaining:")
print(combined.isnull().sum()[combined.isnull().sum() > 0] if combined.isnull().any().any() else "None ✓")

# ============================================
# LOG TRANSFORM FARE (because it is skewed)
# ============================================

combined['fare_log'] = np.log1p(combined['Fare'])
print(f"\nFare log-transformed (new column: fare_log)")
print(f"Original Fare range: {combined['Fare'].min():.2f} to {combined['Fare'].max():.2f}")
print(f"Log Fare range:      {combined['fare_log'].min():.2f} to {combined['fare_log'].max():.2f}")

# ============================================
# ONE-HOT ENCODE CATEGORICAL FEATURES
# ============================================

# Encode Sex (binary - just convert to 0/1)
combined['Sex'] = (combined['Sex'] == 'female').astype(int)
print(f"\nSex encoded: female=1, male=0")

# One-hot encode Embarked and title
combined = pd.get_dummies(combined, columns=['Embarked', 'title'], prefix=['emb', 'title'])

# Convert any boolean columns to integers
bool_cols = combined.select_dtypes(include='bool').columns
combined[bool_cols] = combined[bool_cols].astype(int)

print(f"\nFinal columns after encoding:")
print(combined.columns.tolist())

print(f"\nFinal combined dataset shape: {combined.shape}")




Age missing values filled with median: 28.0
Embarked missing values filled with mode: S
Fare missing values filled with median: 14.4542

Missing values remaining:
None ✓

Fare log-transformed (new column: fare_log)
Original Fare range: 0.00 to 512.33
Log Fare range:      0.00 to 6.24

Sex encoded: female=1, male=0

Final columns after encoding:
['PassengerId', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'source', 'family_size', 'is_alone', 'has_cabin', 'fare_log', 'emb_C', 'emb_Q', 'emb_S', 'title_Master', 'title_Miss', 'title_Mr', 'title_Mrs', 'title_Rare']

Final combined dataset shape: (1309, 20)


In [6]:

print("\nFirst 3 rows of  data:")
display(combined.head(3))


First 3 rows of  data:


,PassengerId,Pclass,Sex,Age,SibSp,Parch,Fare,source,family_size,is_alone,has_cabin,fare_log,emb_C,emb_Q,emb_S,title_Master,title_Miss,title_Mr,title_Mrs,title_Rare
0,1,3,0,22.0,1,0,7.2500,train,2,0,0,2.110213,0,0,1,0,0,1,0,0
1,2,1,1,38.0,1,0,71.2833,train,2,0,1,4.280593,1,0,0,0,0,0,1,0
2,3,3,1,26.0,0,0,7.9250,train,1,1,0,2.188856,0,0,1,0,1,0,0,0


In [7]:
# Cell 6 — Split combined data back and prepare for modelling

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler

# Separate train and test using the source marker
train_processed = combined[combined['source'] == 'train'].copy()
test_processed = combined[combined['source'] == 'test'].copy()

# Drop the source marker — no longer needed
train_processed = train_processed.drop('source', axis=1)
test_processed = test_processed.drop('source', axis=1)

# Save passenger IDs for the test set (needed for Kaggle submission)
test_passenger_ids = test_processed['PassengerId'].copy()

# Drop PassengerId from both — it has no predictive value
train_processed = train_processed.drop('PassengerId', axis=1)
test_processed = test_processed.drop('PassengerId', axis=1)

print(f"Train processed shape: {train_processed.shape}")
print(f"Test processed shape:  {test_processed.shape}")
print(f"\nFinal feature list ({len(train_processed.columns)} features):")
print(train_processed.columns.tolist())

# X_train_full = all training features
# y_train = target (we saved this earlier)
X_train_full = train_processed
X_test_kaggle = test_processed  # this is what we predict on for Kaggle submission

# Split train_processed into our own train and validation sets
X_train, X_val, y_train_split, y_val = train_test_split(
    X_train_full, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

print(f"\nTraining split:    {X_train.shape[0]} rows")
print(f"Validation split:  {X_val.shape[0]} rows")
print(f"Kaggle test:       {X_test_kaggle.shape[0]} rows")

# Verify class balance preserved
print(f"\nClass balance:")
print(f"Full train survival rate:    {y_train.mean()*100:.1f}%")
print(f"Train split survival rate:   {y_train_split.mean()*100:.1f}%")
print(f"Validation survival rate:    {y_val.mean()*100:.1f}%")

Train processed shape: (891, 18)
Test processed shape:  (418, 18)

Final feature list (18 features):
['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'family_size', 'is_alone', 'has_cabin', 'fare_log', 'emb_C', 'emb_Q', 'emb_S', 'title_Master', 'title_Miss', 'title_Mr', 'title_Mrs', 'title_Rare']

Training split:    712 rows
Validation split:  179 rows
Kaggle test:       418 rows

Class balance:
Full train survival rate:    38.4%
Train split survival rate:   38.3%
Validation survival rate:    38.5%


In [8]:
# Cell 7 — Train Logistic Regression model

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Step 1: Scale features (Logistic Regression needs scaled features)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_kaggle_scaled = scaler.transform(X_test_kaggle)

print("Features scaled ✓")

# Step 2: Train model
print("\n" + "=" * 60)
print("TRAINING LOGISTIC REGRESSION")
print("=" * 60)

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train_split)

print("Model trained ✓")

# Step 3: Evaluate on validation set
print("\n" + "=" * 60)
print("VALIDATION RESULTS — LOGISTIC REGRESSION")
print("=" * 60)

lr_val_predictions = lr_model.predict(X_val_scaled)

print(f"Accuracy:  {accuracy_score(y_val, lr_val_predictions):.3f}")
print(f"Precision: {precision_score(y_val, lr_val_predictions):.3f} (for survived class)")
print(f"Recall:    {recall_score(y_val, lr_val_predictions):.3f} (for survived class)")
print(f"F1 Score:  {f1_score(y_val, lr_val_predictions):.3f}")

print("\n" + "=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)
cm = confusion_matrix(y_val, lr_val_predictions)
print(f"                Predicted DIED   Predicted SURVIVED")
print(f"Actual DIED          {cm[0][0]}                {cm[0][1]}")
print(f"Actual SURVIVED      {cm[1][0]}                {cm[1][1]}")

# Step 4: Cross-validation for more reliable estimate
print("\n" + "=" * 60)
print("5-FOLD CROSS-VALIDATION")
print("=" * 60)

X_train_full_scaled = scaler.fit_transform(X_train_full)
cv_scores = cross_val_score(
    LogisticRegression(max_iter=1000, random_state=42),
    X_train_full_scaled,
    y_train,
    cv=5,
    scoring='accuracy'
)

print(f"CV Scores: {cv_scores.round(3)}")
print(f"Mean accuracy: {cv_scores.mean():.3f}")
print(f"Std deviation: {cv_scores.std():.3f}")

# Step 5: Feature importance
print("\n" + "=" * 60)
print("TOP 10 MOST INFLUENTIAL FEATURES")
print("=" * 60)
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': lr_model.coef_[0]
})
feature_importance['Abs_Coef'] = feature_importance['Coefficient'].abs()
feature_importance = feature_importance.sort_values('Abs_Coef', ascending=False)
print(feature_importance[['Feature', 'Coefficient']].head(10).to_string(index=False))

print("\nPositive coefficient → pushes prediction toward SURVIVED")
print("Negative coefficient → pushes prediction toward DIED")

Features scaled ✓

TRAINING LOGISTIC REGRESSION
Model trained ✓

VALIDATION RESULTS — LOGISTIC REGRESSION
Accuracy:  0.838
Precision: 0.794 (for survived class)
Recall:    0.783 (for survived class)
F1 Score:  0.788

CONFUSION MATRIX
                Predicted DIED   Predicted SURVIVED
Actual DIED          96                14
Actual SURVIVED      15                54

5-FOLD CROSS-VALIDATION
CV Scores: [0.832 0.803 0.798 0.815 0.876]
Mean accuracy: 0.825
Std deviation: 0.028

TOP 10 MOST INFLUENTIAL FEATURES
     Feature  Coefficient
         Sex     0.844986
    fare_log     0.547709
    title_Mr    -0.436350
      Pclass    -0.432522
title_Master     0.427739
       SibSp    -0.404496
         Age    -0.388137
   has_cabin     0.372579
 family_size    -0.363107
   title_Mrs     0.306951

Positive coefficient → pushes prediction toward SURVIVED
Negative coefficient → pushes prediction toward DIED


In [9]:
# Cell 8 — Train Random Forest model

from sklearn.ensemble import RandomForestClassifier

print("=" * 60)
print("TRAINING RANDOM FOREST")
print("=" * 60)

# Random Forest does NOT need feature scaling
# Use original X_train, X_val (not scaled versions)

rf_model = RandomForestClassifier(
    n_estimators=200,       # number of trees
    max_depth=8,            # maximum depth per tree
    min_samples_split=5,    # minimum samples to split a node
    random_state=42,
    n_jobs=-1               # use all CPU cores
)

rf_model.fit(X_train, y_train_split)
print("Model trained ✓")
print(f"Number of trees: {rf_model.n_estimators}")

# Evaluate on validation set
print("\n" + "=" * 60)
print("VALIDATION RESULTS — RANDOM FOREST")
print("=" * 60)

rf_val_predictions = rf_model.predict(X_val)

print(f"Accuracy:  {accuracy_score(y_val, rf_val_predictions):.3f}")
print(f"Precision: {precision_score(y_val, rf_val_predictions):.3f}")
print(f"Recall:    {recall_score(y_val, rf_val_predictions):.3f}")
print(f"F1 Score:  {f1_score(y_val, rf_val_predictions):.3f}")

print("\n" + "=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)
cm = confusion_matrix(y_val, rf_val_predictions)
print(f"                Predicted DIED   Predicted SURVIVED")
print(f"Actual DIED          {cm[0][0]}                {cm[0][1]}")
print(f"Actual SURVIVED      {cm[1][0]}                {cm[1][1]}")

# Cross-validation
print("\n" + "=" * 60)
print("5-FOLD CROSS-VALIDATION")
print("=" * 60)

cv_scores_rf = cross_val_score(
    RandomForestClassifier(n_estimators=200, max_depth=8,
                           min_samples_split=5, random_state=42, n_jobs=-1),
    X_train_full,
    y_train,
    cv=5,
    scoring='accuracy'
)

print(f"CV Scores: {cv_scores_rf.round(3)}")
print(f"Mean accuracy: {cv_scores_rf.mean():.3f}")
print(f"Std deviation: {cv_scores_rf.std():.3f}")

# Feature importance
print("\n" + "=" * 60)
print("TOP 10 MOST IMPORTANT FEATURES — RANDOM FOREST")
print("=" * 60)
rf_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)
print(rf_importance.head(10).to_string(index=False))

TRAINING RANDOM FOREST
Model trained ✓
Number of trees: 200

VALIDATION RESULTS — RANDOM FOREST
Accuracy:  0.810
Precision: 0.787
Recall:    0.696
F1 Score:  0.738

CONFUSION MATRIX
                Predicted DIED   Predicted SURVIVED
Actual DIED          97                13
Actual SURVIVED      21                48

5-FOLD CROSS-VALIDATION
CV Scores: [0.855 0.809 0.854 0.809 0.843]
Mean accuracy: 0.834
Std deviation: 0.021

TOP 10 MOST IMPORTANT FEATURES — RANDOM FOREST
    Feature  Importance
        Sex    0.172294
   title_Mr    0.156863
   fare_log    0.111114
       Fare    0.110571
        Age    0.088883
     Pclass    0.067651
family_size    0.056557
  has_cabin    0.052023
 title_Miss    0.044148
  title_Mrs    0.042797


In [10]:
# Cell 9 — Train XGBoost model

import xgboost as xgb

print("=" * 60)
print("TRAINING XGBOOST")
print("=" * 60)

xgb_model = xgb.XGBClassifier(
    n_estimators=200,        # boosting rounds
    max_depth=4,             # smaller trees than RF
    learning_rate=0.1,       # how aggressively each tree learns
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)

xgb_model.fit(X_train, y_train_split)
print("Model trained ✓")

# Evaluate on validation set
print("\n" + "=" * 60)
print("VALIDATION RESULTS — XGBOOST")
print("=" * 60)

xgb_val_predictions = xgb_model.predict(X_val)

print(f"Accuracy:  {accuracy_score(y_val, xgb_val_predictions):.3f}")
print(f"Precision: {precision_score(y_val, xgb_val_predictions):.3f}")
print(f"Recall:    {recall_score(y_val, xgb_val_predictions):.3f}")
print(f"F1 Score:  {f1_score(y_val, xgb_val_predictions):.3f}")

print("\n" + "=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)
cm = confusion_matrix(y_val, xgb_val_predictions)
print(f"                Predicted DIED   Predicted SURVIVED")
print(f"Actual DIED          {cm[0][0]}                {cm[0][1]}")
print(f"Actual SURVIVED      {cm[1][0]}                {cm[1][1]}")

# Cross-validation
print("\n" + "=" * 60)
print("5-FOLD CROSS-VALIDATION")
print("=" * 60)

cv_scores_xgb = cross_val_score(
    xgb.XGBClassifier(n_estimators=200, max_depth=4,
                      learning_rate=0.1, random_state=42,
                      eval_metric='logloss', use_label_encoder=False),
    X_train_full,
    y_train,
    cv=5,
    scoring='accuracy'
)

print(f"CV Scores: {cv_scores_xgb.round(3)}")
print(f"Mean accuracy: {cv_scores_xgb.mean():.3f}")
print(f"Std deviation: {cv_scores_xgb.std():.3f}")

# Feature importance
print("\n" + "=" * 60)
print("TOP 10 MOST IMPORTANT FEATURES — XGBOOST")
print("=" * 60)
xgb_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)
print(xgb_importance.head(10).to_string(index=False))

TRAINING XGBOOST


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [13:41:16] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Model trained ✓

VALIDATION RESULTS — XGBOOST
Accuracy:  0.827
Precision: 0.788
Recall:    0.754
F1 Score:  0.770

CONFUSION MATRIX
                Predicted DIED   Predicted SURVIVED
Actual DIED          96                14
Actual SURVIVED      17                52

5-FOLD CROSS-VALIDATION


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [13:41:17] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [13:41:18] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [13:41:19] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [13:41:21] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:

CV Scores: [0.86  0.803 0.876 0.809 0.837]
Mean accuracy: 0.837
Std deviation: 0.028

TOP 10 MOST IMPORTANT FEATURES — XGBOOST
    Feature  Importance
   title_Mr    0.520463
     Pclass    0.102534
 title_Rare    0.102249
  has_cabin    0.068981
family_size    0.043340
      emb_S    0.023739
       Fare    0.023089
      SibSp    0.018411
        Age    0.018006
 title_Miss    0.015629


In [12]:
# Cell 10 — Train final XGBoost on full training data and create submission

print("=" * 60)
print("TRAINING FINAL MODEL ON FULL TRAINING DATA")
print("=" * 60)

# Train on ALL training data, not just the 80% split
final_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

final_model.fit(X_train_full, y_train)
print(f"Model trained on {len(X_train_full)} rows ✓")

# Predict on Kaggle test set
print("\n" + "=" * 60)
print("PREDICTING ON KAGGLE TEST SET")
print("=" * 60)
test_predictions = final_model.predict(X_test_kaggle)
print(f"Predictions made for {len(test_predictions)} passengers ✓")
print(f"Predicted survivors: {test_predictions.sum()}")
print(f"Predicted deaths:    {len(test_predictions) - test_predictions.sum()}")
print(f"Predicted survival rate: {test_predictions.mean()*100:.1f}%")

# Create submission file in Kaggle's required format
submission = pd.DataFrame({
    'PassengerId': test_passenger_ids,
    'Survived': test_predictions
})

submission.to_csv('Titanic_submission.csv', index=False)
print("\n" + "=" * 60)
print("SUBMISSION FILE CREATED")
print("=" * 60)
print("File saved as: Titanic_submission.csv")
print("\nFirst 10 rows:")
print(submission.head(10))
print(f"\nShape: {submission.shape}")

TRAINING FINAL MODEL ON FULL TRAINING DATA
Model trained on 891 rows ✓

PREDICTING ON KAGGLE TEST SET
Predictions made for 418 passengers ✓
Predicted survivors: 149
Predicted deaths:    269
Predicted survival rate: 35.6%

SUBMISSION FILE CREATED
File saved as: Titanic_submission.csv

First 10 rows:
     PassengerId  Survived
891          892         0
892          893         0
893          894         0
894          895         0
895          896         1
896          897         0
897          898         0
898          899         0
899          900         1
900          901         0

Shape: (418, 2)
